https://github.com/kris-nader/scTherapy

r-base

In [1]:
# # Install packages if you haven't already
# install.packages("readxl")

# # Load the libraries
# library(readxl)

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



for the integrated dataset of all samples

In [1]:
my_data <- read.csv("malignant_cells_DEG_all_samples.csv", row.names = 1)
saveRDS(my_data, file = "malignant_cells_DEG_all_samples.RDS")

In [2]:
# load data
malignant_cells_DEG <- readRDS("malignant_cells_DEG_all_samples.RDS")

head(malignant_cells_DEG)
#           avg_log2FC    p_val_adj
# LYZ       5.254787      1.512117e-107
# S100A9    4.530434      9.073954e-39
# 	         .............
# IL32 	   -3.599663      3.197208e-164
# GNLY     -4.417322      8.259763e-81

,avg_log2FC,p_val_adj
,<dbl>,<dbl>
CASC15,5.694498,0
SOX4,5.247437,0
GNAQ,3.113246,0
SSBP2,3.817840,0
CDK6,3.808591,0
MED13L,2.274940,0


In [8]:
install.packages(c("dplyr", "logger", "httr", "jsonlite", "readr"))

also installing the dependencies ‘progress’, ‘cpp11’, ‘vctrs’, ‘hms’, ‘vroom’, ‘tzdb’


Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [3]:
# load libraries
invisible(lapply(c("dplyr","logger","httr", "jsonlite", "readr"), library, character.only = !0));
invisible(source("https://raw.githubusercontent.com/kris-nader/scTherapy/main/R/predict_compounds.R"))

# load data for making drug:dose predictions
gene_list <- "https://raw.githubusercontent.com/kris-nader/scTherapy/main/geneinfo_beta_input.txt"
gene_info <- data.table::fread(gene_list) %>% as.data.frame()

# filter DEG
DEG_malignant <- malignant_cells_DEG %>%
  mutate(gene_symbol = rownames(.)) %>% inner_join(gene_info, by = "gene_symbol") %>%
  filter((p_val_adj <= 0.05 & (avg_log2FC > 1 | avg_log2FC < -1)) | (avg_log2FC > -0.1 & avg_log2FC < 0.1))
DEG_malignant_list <- setNames(as.list(DEG_malignant$avg_log2FC), DEG_malignant$gene_symbol)

#predict monotherapies for malignant cluster
monotherapy_drugs <- predict_drugs(DEG_malignant_list)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [4]:
monotherapy_drugs

,cid,Drug_Name,Response,MoA,Dose.nM.,Toxicity
,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>
1,6918837,panobinostat,High,HDAC_inhibitor,730.0000,Potentially toxic (low confidence)
2,135564985,ganetespib,High-to-moderate,-,622.4787,Potentially toxic (low confidence)
3,6505803,tanespimycin,High-to-moderate,HSP_inhibitor,281.4812,
4,53358942,idasanutlin,High-to-moderate,MDM_inhibitor,438.7463,
5,11364421,BI-2536,High-to-moderate,PLK_inhibitor,498.9539,
6,30323,daunorubicin,High-to-moderate,RNA_synthesis_inhibitor_topoisomerase_inhibitor,1000.0000,
7,6914657,FK-866,High-to-moderate,niacinamide_phosphoribosyltransferase_inhibitor,389.5792,
8,25227462,PF-03758309,High-to-moderate,p21_activated_kinase_inhibitor,483.1389,Potentially toxic (low confidence)
9,24800541,delanzomib,High-to-moderate,proteasome_inhibitor,240.5565,Potentially toxic (low confidence)


In [5]:
write.csv(monotherapy_drugs, "scTherapy_monotherapy_drugs_all_samples2.csv", row.names = TRUE)

install.packages("writexl")   
library(writexl)
writexl::write_xlsx(monotherapy_drugs, "scTherapy_monotherapy_drugs_all_samples2.xlsx")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



per each sample

In [7]:
samples <- c("ak", "az", "b", "v", "ts")

invisible(lapply(c("dplyr","logger","httr", "jsonlite", "readr"), library, character.only = !0))
invisible(source("https://raw.githubusercontent.com/kris-nader/scTherapy/main/R/predict_compounds.R"))

gene_list <- "https://raw.githubusercontent.com/kris-nader/scTherapy/main/geneinfo_beta_input.txt"
gene_info <- data.table::fread(gene_list) %>% as.data.frame()

library(writexl)

for (sample in samples) {
  input_csv <- paste0("malignant_cells_DEG_", sample, "_tcells.csv")
  my_data <- read.csv(input_csv, row.names = 1)
  saveRDS(my_data, file = paste0("malignant_cells_DEG_", sample, "_tcells.RDS"))
  
  malignant_cells_DEG <- my_data
  
  DEG_malignant <- malignant_cells_DEG %>%
    mutate(gene_symbol = rownames(.)) %>%
    inner_join(gene_info, by = "gene_symbol") %>%
    filter((p_val_adj <= 0.05 & (avg_log2FC > 1 | avg_log2FC < -1)) | (avg_log2FC > -0.1 & avg_log2FC < 0.1))
  
  DEG_malignant_list <- setNames(as.list(DEG_malignant$avg_log2FC), DEG_malignant$gene_symbol)
  monotherapy_drugs <- predict_drugs(DEG_malignant_list)
  
  write.csv(monotherapy_drugs, file = paste0("scTherapy_monotherapy_drugs_", sample, ".csv"), row.names = TRUE)
  write_xlsx(monotherapy_drugs, path = paste0("scTherapy_monotherapy_drugs_", sample, ".xlsx"))
}